In [ ]:
# ==========================================================
# ADIM 1: Kurulum ve Kütüphaneler
# ==========================================================
# Hocanın koduyla aynı
!pip install transformers datasets -q

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

print("Adım 1: Gerekli kütüphaneler (transformers, torch, pandas) yüklendi.")


# ==========================================================
# ADIM 2: Veri Yükleme ve Hazırlama (SENİN VERİ SETİN)
# ==========================================================
# Bu bölüm, hocanın 'load_dataset("imdb")' kısmını
# senin 'clean_data222.csv' dosyanla değiştirir.

yerel_dosya_yolu = "clean_data.csv"
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'label'

try:
    print(f"\nAdım 2: '{yerel_dosya_yolu}' dosyasından veri seti yükleniyor...")
    df = pd.read_csv(yerel_dosya_yolu)

    # 'clean_content' sütununu 'text' olarak yeniden adlandıralım
    if 'clean_content' in df.columns:
        df.rename(columns={'clean_content': 'text'}, inplace=True)

    print(f"Veri boyutu (filtresiz): {df.shape}")

    # 1. Etiketleri Metinden Tamsayıya Dönüştürme (6 Sınıf için)
    # BERT modeli de CNN gibi 'Politics' yerine 0, 1, 2... bekler
    labels_list = sorted(df[LABEL_COLUMN].unique())
    label_to_int = {label: i for i, label in enumerate(labels_list)}
    # Tahmin fonksiyonunda kullanmak için tersini de oluşturalım
    int_to_label = {i: label for label, i in label_to_int.items()}

    df['label_int'] = df[LABEL_COLUMN].map(label_to_int)

    # Bu değişkenler sonraki adımlarda kullanılacak
    num_classes = len(label_to_int) # Toplam sınıf sayısı (6)

    print(f"\n{num_classes} sınıf bulundu ve sayısallaştırıldı:")
    print(label_to_int)
    print("-" * 30)

    # 2. Veriyi Eğitim ve Test Setlerine Ayırma
    X = df[TEXT_COLUMN].fillna('').values # NaN hatalarını önle
    y = df['label_int'].values

    # stratify=y: 6 sınıfın oranını korur (CNN'deki gibi)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    print(f"Eğitim Seti (X_train) Boyutu: {len(X_train)}")
    print(f"Test Seti (X_test) Boyutu: {len(X_test)}")
    print(f"Veri hazırlama (Adım 2) tamamlandı.")
    print("\n--- Adım 1 ve 2 tamamlandı. Şimdi Adım 3'e (Dataset Sınıfı) geçebilirsiniz. ---")


except Exception as e:
    print(f"\n\nHATA: Veri işlenirken bir sorun oluştu: {e}")

In [ ]:
# ==========================================================
# ADIM 3 (Güncellenmiş - Import Eklendi): DistilBERT Tokenizer ve Dataset
# ==========================================================
# 'NameError' hatasını düzeltmek için import'u buraya tekrar ekliyoruz.
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

try:
    # 1. DistilBERT Tokenizer'ı Yükleme
    # Slayt 41'de önerilen modeli kullanıyoruz
    MODEL_NAME = 'distilbert-base-uncased'

    print(f"Adım 3: DistilBERT Tokenizer ('{MODEL_NAME}') yükleniyor...")
    # BERT yerine DistilBertTokenizer kullanıyoruz
    tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
    print("Tokenizer başarıyla yüklendi.")


    # 2. Dataset Sınıfını Tanımlama
    class NewsDistilBertDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len=256):
            self.texts = texts
            self.labels = labels
            self.tokenizer = tokenizer
            # max_len=256, GPU'da (T4) verimli çalışır
            self.max_len = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            text = str(self.texts[idx])
            label = self.labels[idx]

            # Tokenizer'ı kullanarak metni DistilBERT formatına dönüştür
            encoding = self.tokenizer(
                text,
                add_special_tokens=True, # [CLS] ve [SEP] ekle
                max_length=self.max_len, # max_len'e göre doldur/kırp
                padding='max_length',    # 'max_length'e kadar doldur
                truncation=True,         # 'max_length'ten uzunsa kırp
                return_tensors='pt'      # PyTorch tensörü olarak döndür
            )

            # Hocanın koduyla aynı formatta bir sözlük (dictionary) döndür
            return {
                'input_ids': encoding['input_ids'].flatten(),
                'attention_mask': encoding['attention_mask'].flatten(),
                'labels': torch.tensor(label, dtype=torch.long)
            }

    # --- Test ---
    print("\nNewsDistilBertDataset sınıfı oluşturuluyor...")

    MAX_LEN = 256

    # Adım 1+2'den gelen X_train, y_train vb. kullanılır.
    train_dataset_bert = NewsDistilBertDataset(X_train, y_train, tokenizer, max_len=MAX_LEN)
    test_dataset_bert = NewsDistilBertDataset(X_test, y_test, tokenizer, max_len=MAX_LEN)

    print(f"Dataset'ler MAX_LEN={MAX_LEN} ile oluşturuldu.")

    # Bir adet örnek alıp bakalım:
    sample = train_dataset_bert[0]
    print(f"\nEğitim setinden ilk örnek (DistilBERT formatında):")
    print(f"  input_ids shape: {sample['input_ids'].shape}") # [256] olmalı
    print(f"  attention_mask shape: {sample['attention_mask'].shape}") # [256] olmalı
    print(f"  label: {sample['labels']}") # 0-5 arası bir sayı olmalı

    print("\n--- Adım 3 tamamlandı. Şimdi Adım 4'e (DataLoader) geçebilirsiniz. ---")

except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Bu bloğu çalıştırmadan önce Adım 1+2 bloğunu başarıyla çalıştırdığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")

In [ ]:
# ==========================================================
# ADIM 4: DataLoader'ların Oluşturulması
# ==========================================================
# Bu adım, Adım 3'te oluşturulan 'train_dataset_bert' ve 'test_dataset_bert'
# değişkenlerini kullanır.

try:
    # Mini-grup (batch) boyutu.
    # DistilBERT daha küçük olduğu için 16'dan daha büyük (örn: 32)
    # bir BATCH_SIZE deneyebiliriz, bu GPU'da eğitimi hızlandırır.
    BATCH_SIZE = 32

    # Eğitim verisi için DataLoader
    # dataset=train_dataset_bert: Adım 3'te oluşturduğumuz sınıfı kullanır.
    # shuffle=True: Her eğitim turunda (epoch) veriyi karıştırır.
    train_loader_bert = DataLoader(
        dataset=train_dataset_bert,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    # Test verisi için DataLoader
    # dataset=test_dataset_bert: Adım 3'te oluşturduğumuz sınıfı kullanır.
    # shuffle=False: Test ederken veriyi karıştırmaya gerek yoktur.
    test_loader_bert = DataLoader(
        dataset=test_dataset_bert,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    print(f"Adım 4: DataLoader'lar BATCH_SIZE={BATCH_SIZE} ile başarıyla oluşturuldu.")
    print(f"Eğitim DataLoader'ında yaklaşık {len(train_loader_bert)} mini-grup (batch) var.")
    print(f"Test DataLoader'ında yaklaşık {len(test_loader_bert)} mini-grup (batch) var.")

    # --- Test ---
    # Bir mini-grubu (batch) çekip boyutlarını kontrol edelim
    print("\nBir eğitim mini-grubu (batch) test ediliyor...")
    data_iter_bert = iter(train_loader_bert)
    batch = next(data_iter_bert)

    # Dataset'ten dönen sözlük formatını kontrol et
    print(f"  input_ids boyutu: {batch['input_ids'].shape}")
    print(f"  attention_mask boyutu: {batch['attention_mask'].shape}")
    print(f"  labels boyutu: {batch['labels'].shape}")

    print(f"(Beklenen input_ids boyutu: [{BATCH_SIZE}, {MAX_LEN}])")
    print(f"(Beklenen labels boyutu: [{BATCH_SIZE}])")

    print("\n--- Adım 4 tamamlandı. Şimdi Adım 5'e (Modeli Başlatma) geçebilirsiniz. ---")


except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Bu bloğu çalıştırmadan önce Adım 1+2 ve Adım 3 bloklarını başarıyla çalıştırdığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")

In [ ]:
# ==========================================================
# ADIM 5: Modelin (DistilBERT) ve Optimize Edicinin Başlatılması
# ==========================================================
# Bu adım, 'BertForSequenceClassification' yerine
# 'DistilBertForSequenceClassification' modelini başlatır.
# Adım 1+2'deki 'num_classes' (6 olan) değişkenini kullanır.

# NameError yaşamamak için import'u tekrar ekleyelim
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW

try:
    # 1. Cihazı (Device) Ayarlama
    # T4 GPU'yu seçtiğiniz için bu 'cuda' olmalı
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Adım 5: Model '{device}' cihazı üzerinde başlatılacak.")

    # Eğer 'cpu' yazarsa, lütfen Adım 1'den önceki GPU
    # etkinleştirme adımını ("Çalışma zamanı türünü değiştir") tekrar kontrol edin.

    # 2. Modeli Yükleme
    # Slayt 41'deki önerilen modeli kullanıyoruz
    MODEL_NAME = 'distilbert-base-uncased'

    model_bert = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        # EN ÖNEMLİ DEĞİŞİKLİK: Sınıf sayısını 6 olarak ayarlıyoruz
        num_labels=num_classes  # 'num_classes' Adım 1+2'de 6 olarak hesaplanmıştı
    )

    # Modeli 'cuda' (GPU) cihazına taşı
    model_bert = model_bert.to(device)
    print(f"Model ('{MODEL_NAME}') {num_classes} etiket (sınıf) için başarıyla yüklendi.")

    # 3. Optimize Ediciyi (Optimizer) Başlatma
    # AdamW, Transformer modelleri (Slayt 33) için standart optimize edicidir.
    optimizer_bert = AdamW(model_bert.parameters(), lr=2e-5)

    print("Optimize Edici (AdamW) başarıyla tanımlandı.")
    print("\n--- Adım 5 tamamlandı. Şimdi Adım 6'ya (Eğitim Fonksiyonları) geçebilirsiniz. ---")


except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Bu bloğu çalıştırmadan önce Adım 1+2 bloğunu başarıyla çalıştırdığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")

In [ ]:
# ==========================================================
# ADIM 6: Eğitim ve Değerlendirme Fonksiyonları
# ==========================================================
# Bu bölüm, 'train_epoch' ve 'evaluate' fonksiyonlarını tanımlar.
# Bu fonksiyonlar Adım 7'deki ana eğitim döngüsünde kullanılacaktır.
# Bu fonksiyonlar Adım 4'teki '...loader_bert' ve Adım 5'teki
# 'model_bert', 'optimizer_bert', 'device' değişkenlerini kullanır.

try:
    # --- Eğitim Fonksiyonu ---
    def train_epoch(model, loader, optimizer, device):
        model.train() # Modeli "eğitim modu"na al
        total_loss = 0
        correct = 0
        total = 0

        # Adım 4'te oluşturulan 'train_loader_bert'i kullan
        for batch in tqdm(loader, desc="Eğitim Adımı (Training)"):
            # Veri grubundaki (batch) tensörleri al
            # ve 'cuda' cihazına gönder
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Gradyanları sıfırla
            optimizer.zero_grad()

            # Modeli çalıştır (Forward pass)
            # 'labels' parametresini verdiğimizde model kaybı (loss) da hesaplar
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            # Kaybı (loss) al
            # DistilBERT de BERT gibi 'loss' ve 'logits' döndürür
            loss = outputs.loss

            # Geriye yayılım (Backpropagation)
            loss.backward()
            optimizer.step()

            # İstatistikleri kaydet
            total_loss += loss.item()
            predictions = torch.argmax(outputs.logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        # Epoch sonu ortalamaları
        return total_loss / len(loader), correct / total

    # --- Değerlendirme Fonksiyonu ---
    def evaluate(model, loader, device):
        model.eval() # Modeli "değerlendirme modu"na al (Dropout'u kapatır)
        total_loss = 0
        correct = 0
        total = 0

        # Gradyan hesaplamasını durdur
        with torch.no_grad():
            # Adım 4'te oluşturulan 'test_loader_bert'i kullan
            for batch in tqdm(loader, desc="Değerlendirme (Evaluating)"):
                # Veriyi 'cuda' cihazına gönder
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                # Modeli çalıştır
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                # Kaybı ve istatistikleri kaydet
                loss = outputs.loss
                total_loss += loss.item()
                predictions = torch.argmax(outputs.logits, dim=1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)

        # Epoch sonu ortalamaları
        return total_loss / len(loader), correct / total

    print("Adım 6: 'train_epoch' ve 'evaluate' fonksiyonları başarıyla tanımlandı.")
    print("\n--- Adım 6 tamamlandı. Şimdi Adım 7'ye (Ana Eğitim Döngüsü) geçebilirsiniz. ---")

except Exception as e:
    print(f"\nBir hata oluştu: {e}")

In [ ]:
# ==========================================================
# ADIM 7: Ana Eğitim Döngüsü
# ==========================================================
# Bu adım, '===== TRAIN MODEL =====' bölümüne karşılık gelir.
# Adım 6'da tanımlanan 'train_epoch' ve 'evaluate'
# fonksiyonlarını çağırır.

try:
    # Epoch sayısı. DistilBERT ince ayarı (fine-tuning) için 3-4
    # epoch genellikle iyi sonuç verir.
    EPOCHS = 3

    print(f"Adım 7: Ana eğitim döngüsü başlıyor... Toplam {EPOCHS} epoch sürecek.")
    print(f"Kullanılan cihaz: {device}. (GPU ile bu hızlı olmalı)")

    best_test_accuracy = 0 # En iyi modeli takip etmek için

    for epoch in range(EPOCHS):
        print(f"\n===== Epoch {epoch + 1}/{EPOCHS} =====")

        # Modeli eğit (Adım 6'daki fonksiyon)
        train_loss, train_acc = train_epoch(
            model_bert,
            train_loader_bert,
            optimizer_bert,
            device
        )

        print(f"Epoch {epoch + 1} Eğitim Sonucu:")
        print(f"  Ortalama Eğitim Kaybı (Loss): {train_loss:.4f}")
        print(f"  Eğitim Doğruluğu (Accuracy): {train_acc * 100:.2f}%")

        # Modeli değerlendir (Adım 6'daki fonksiyon)
        test_loss, test_acc = evaluate(
            model_bert,
            test_loader_bert,
            device
        )

        print(f"Epoch {epoch + 1} Değerlendirme Sonucu:")
        print(f"  Ortalama Test Kaybı (Loss): {test_loss:.4f}")
        print(f"  Test Doğruluğu (Accuracy): {test_acc * 100:.2f}%")

        # ==================================================
        # Checkpointing: En iyi modeli kaydet
        # ==================================================
        if test_acc > best_test_accuracy:
            best_test_accuracy = test_acc
            checkpoint_path = "distilbert_best_model.pt"
            print(f"\n  Yeni en iyi Test Doğruluğu! Model '{checkpoint_path}' olarak kaydediliyor...")

            # Modeli kaydet (hocanın kodundaki gibi)
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model_bert.state_dict(),
                'optimizer_state_dict': optimizer_bert.state_dict(),
                'loss': test_loss,
                'accuracy': test_acc
            }, checkpoint_path)
        # ==================================================


    print(f"\n--- Adım 7 (Eğitim) {EPOCHS} epoch için tamamlandı. ---")
    print(f"Eğitim boyunca elde edilen en iyi Test Doğruluğu: {best_test_accuracy * 100:.2f}%")
    print("Şimdi Adım 8'e (Tahmin Fonksiyonu) geçebilirsiniz.")


except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Bu bloğu çalıştırmadan önce Adım 4, 5 ve 6 bloklarını başarıyla çalıştırdığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")

In [ ]:
# ==========================================================
# ADIM 8: Tahmin Fonksiyonunun Tanımlanması
# ==========================================================
# Bu adım, '===== PREDICTION FUNCTION =====' bölümüne karşılık gelir.
# Yeni metinleri alıp, eğitilmiş 'model_bert' ile tahmin yapmak için kullanılır.
# Adım 1+2'deki 'int_to_label' (0 -> 'Emotion' vb.) ve
# Adım 3'teki 'tokenizer' ve 'MAX_LEN' değişkenlerini kullanır.

try:
    # (Gerekli olabilecek importları tekrar ekleyelim)
    import torch
    import torch.nn.functional as F

    def predict_category(text, model, tokenizer, device, int_to_label_map, max_len=256):
        # Modeli "değerlendirme modu"na al (Dropout'u kapatır)
        model.eval()

        # Metni, modelin anladığı formata (input_ids, attention_mask) dönüştür
        encoding = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Tensörleri 'cuda' cihazına gönder
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)

        # Gradyan hesaplamasını durdur
        with torch.no_grad():
            # Modeli çalıştır
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            # Çıktıları olasılıklara dönüştür (Softmax)
            probabilities = F.softmax(outputs.logits, dim=1)

            # En yüksek olasılığa sahip sınıfı (indeks olarak) al
            prediction_idx = torch.argmax(probabilities, dim=1).item()

            # Güven skorunu (olasılığı) al
            confidence = probabilities[0][prediction_idx].item()

            # İndeksi (örn: 2) metin etiketine (örn: 'Health') çevir
            category = int_to_label_map[prediction_idx]

        return category, confidence

    print("Adım 8: 'predict_category' fonksiyonu başarıyla tanımlandı.")
    print("\n--- Adım 8 tamamlandı. ---")
    print("Modeli en iyi performansta (Epoch 2) kullanmak için Adım 9'u (Model Yükleme) çalıştırın.")


except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Lütfen önceki adımları (Adım 1+2, Adım 3, Adım 5) çalıştırdığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")

In [ ]:
# ==========================================================
# ADIM 9: En İyi Modelin Yüklenmesi ve Test Tahminleri
# ==========================================================
# Bu adım, "hocanın kodundaki" '===== TEST PREDICTIONS ====='
# bölümüne karşılık gelir.
# Adım 7'de kaydedilen en iyi modeli ('distilbert_best_model.pt')
# yükler ve Adım 8'de tanımlanan 'predict_category' fonksiyonunu kullanır.

try:
    print("Adım 9: En iyi model ('distilbert_best_model.pt') yükleniyor...")

    # 1. Modeli Yeniden Başlat (Boş bir yapı olarak)
    # Adım 5'teki gibi boş bir model mimarisi oluştur
    MODEL_NAME = 'distilbert-base-uncased'

    loaded_model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_classes # 6 sınıf
    )

    # 2. Kaydedilen Ağırlıkları Yükle
    # Kayıt dosyasını 'cuda' (GPU) veya 'cpu' üzerine yükle
    checkpoint = torch.load('distilbert_best_model.pt', map_location=device)

    # Modeli 'cuda' cihazına gönder
    loaded_model = loaded_model.to(device)

    # Kayıtlı 'model_state_dict'i boş modele uygula
    loaded_model.load_state_dict(checkpoint['model_state_dict'])

    print("En iyi model (Epoch 2, Test Acc: 93.77%) başarıyla yüklendi.")
    print("\n" + "="*50 + "\n")


    # ==========================================================
    # YENİ METİNLER ÜZERİNDE TEST
    # ==========================================================
    # Buraya kendi 6 sınıfınıza uygun örnekler yazabilirsiniz:

    test_texts_list = [
        # Politics
        "The prime minister announced new election dates yesterday.",
        # Sport
        "What a goal! The striker scored in the final minute of the match.",
        # Health
        "Doctors recommend a balanced diet and regular exercise to reduce heart disease risk.",
        # Financial
        "The stock market reacted positively to the new inflation report, with stocks rising.",
        # Emotion
        "I am so incredibly happy and excited about the good news!",
        # Science
        "NASA's new telescope has discovered a planet with potential signs of water."
    ]

    print("Yeni metinler üzerinde tahminler yapılıyor...")

    for review in test_texts_list:
        # Adım 8'de tanımlanan fonksiyonu kullan
        category, confidence = predict_category(
            review,
            loaded_model, # Hafızadaki 'model_bert' yerine 'loaded_model'i kullan
            tokenizer,    # Adım 3'ten
            device,       # Adım 5'ten
            int_to_label  # Adım 1+2'den
        )
        print(f"\nMetin: '{review}'")
        print(f"  -> Tahmin: {category} (Güven: {confidence:.2%})")

    print("\n\n--- DistilBERT süreci tamamlandı. ---")


except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Lütfen önceki tüm adımları (Adım 1+2, 3, 5, 8) çalıştırdığınızdan emin olun.")
except FileNotFoundError:
    print("\n--- HATA ---")
    print("Model dosyası ('distilbert_best_model.pt') bulunamadı.")
    print("Lütfen Adım 7'yi (Eğitim) başarıyla tamamladığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")

In [ ]:
# ==========================================================
# ADIM 10 (Düzeltilmiş): Detaylı Sınıflandırma Raporu
# ==========================================================
# 'object of type 'int' has no len()' hatasını düzeltmek için
# 'target_names' oluşturma satırı güncellendi.

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

try:
    print("Adım 10 (Düzeltilmiş): En iyi model ('distilbert_best_model.pt') yükleniyor...")

    # 1. Modeli Yeniden Başlat (Boş bir yapı olarak)
    MODEL_NAME = 'distilbert-base-uncased'
    loaded_model_for_report = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_classes # 6 sınıf
    )

    # 2. Kaydedilen Ağırlıkları Yükle
    checkpoint = torch.load('distilbert_best_model.pt', map_location=device)
    loaded_model_for_report.load_state_dict(checkpoint['model_state_dict'])
    loaded_model_for_report = loaded_model_for_report.to(device)
    loaded_model_for_report.eval()

    print("En iyi model (Test Acc: 93.77%) yüklendi. Tüm test seti üzerinde tahminler başlıyor...")

    # 3. Tüm Test Seti Üzerinde Tahminleri Topla
    all_predictions = []
    all_true_labels = []

    with torch.no_grad():
        for batch in tqdm(test_loader_bert, desc="Test Verisi Tahminleri"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = loaded_model_for_report(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            predicted = torch.argmax(outputs.logits, dim=1)
            all_predictions.extend(predicted.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    print("Tüm test tahminleri toplandı.")
    print("\n" + "="*50 + "\n")

    # --- Sonuç Raporlaması ---

    # 1. Genel Doğruluk (Accuracy)
    test_accuracy = accuracy_score(all_true_labels, all_predictions)
    print("DistilBERT Performans Raporu (Test Verisi):")
    print(f"Genel Doğruluk (Accuracy): {test_accuracy:.4f} (Tahminlerin %{test_accuracy*100:.2f}'si doğru)")
    print("\n" + "-"*50 + "\n")

    # 2. Detaylı Sınıflandırma Raporu
    # Adım 1+2'deki 'int_to_label' sözlüğünü kullan

    # === DÜZELTİLEN SATIR BURADA ===
    # 'int_to_label.items()' (örn: (0, 'Emotion')) listesini 'item[0]'a (yani 0, 1, 2...) göre sırala
    # ve 'label'ı (örn: 'Emotion') al.
    target_names = [label for i, label in sorted(int_to_label.items(), key=lambda item: item[0])]
    # === DÜZELTME SONU ===

    print(f"Detaylı Sınıflandırma Raporu ({num_classes} Sınıf):")
    print(classification_report(all_true_labels, all_predictions, target_names=target_names))
    print("\n" + "-"*50 + "\n")

    # 3. Karmaşıklık Matrisi (Confusion Matrix)
    print("Karmaşıklık Matrisi (Confusion Matrix):")
    print("(Satırlar: Gerçek Etiket, Sütunlar: Tahmin Edilen Etiket)")

    cm = confusion_matrix(all_true_labels, all_predictions)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.xlabel('Tahmin Edilen Etiket (Predicted)')
    plt.ylabel('Gerçek Etiket (Actual)')
    plt.title('DistilBERT Model - Karmaşıklık Matrisi (Test Verisi)')
    plt.show()

except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Lütfen önceki tüm adımları (Adım 1+2, 3, 4, 5) çalıştırdığınızdan emin olun.")
except FileNotFoundError:
    print("\n--- HATA ---")
    print("Model dosyası ('distilbert_best_model.pt') bulunamadı.")
    print("Lütfen Adım 7'yi (Eğitim) başarıyla tamamladığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")